# EXP-2026-007 E-eval: evaluate E0, E0s or E3m (GPU)

Card `experiments/registry/EXP-2026-007.md` (approved 2026-11-25, decisions 1 to 9, revised after Codex's review). The
same frozen evaluation for every model of training series E.

- **First session (decision 9):** `MODEL = "E0"` and `SMOKE_FIRST = True`. Before the evaluation, a forward and backward
  pass on labelled-only, image-only and mixed microbatches checks the new training path on the real model (finite
  losses, the pure-noise partner at timestep 999). No optimizer step is taken, and the weights' hash is checked before
  and after. The two trainings start only after this session has passed.
- **Held-out patients:** RHUH's 3 accepted validation patients (V1 to V3) and LUMIERE run r2's 4 accepted development
  pairs (V4 to V7). The RHUH test set stays sealed. Generation reads pre-op inputs and the supplied scenario only.
- **Per patient:** recipe W4 on every axial slice that touches the edit region, seeds 11, 22 and 33, scored from the
  saved stacks on the full frame. Also the copy and the scaffold, the guardrail, the drift outside the edit region,
  held-out diffusion loss (image-only condition; on RHUH also the oracle-conditioned one, labelled oracle), W7's kept
  share on RHUH, and the memorization alert.
- **E3m's session** also reads the E0 and E0s evaluation records and prints the frozen decision.
- **Output:** a write-once record and the stacks, uploaded through the Drive API to
  `MyDrive/RHUH-GBM/synthesis/EXP-2026-007/eval/<RUN_TAG>` and confirmed by MD5. The record names patients: keep that
  folder private. The printed summary uses V1 to V7 only.

Runtime: a G4 GPU. About 1 to 1.5 hours (a little more with the smoke). Keep the tab open until the last cell finishes.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-e-eval"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MSF_REPO, MSF_FILE = "JohnWeck/StableDiffusion", "checkpoint-300.pth"
MODEL = "E0"  # "E0" (r2), "E0s" or "E3m"
TRAIN_TAG = None  # for E0s or E3m: the RUN_TAG of its training run, e.g. "E0s_r1"
SMOKE_FIRST = True  # the first session only (MODEL = "E0")
RUN_TAG = f"eval_{MODEL}_r1"  # a new tag for every attempt; a run folder that holds files is refused
EVAL_TAGS = {"E0": None, "E0s": None}  # for E0s: E0's eval RUN_TAG; for E3m: both, e.g. "eval_E0_r1"
DATA_ON_DRIVE = ("RHUH-GBM", "synthesis", "EXP-2026-001")  # from MyDrive; the API follows the shortcut
OUT_BASE, OUT_RUN = ("RHUH-GBM", "synthesis"), ("EXP-2026-007", "eval", RUN_TAG)
LUMIERE_RUN = ("LUMIERE_derived", "EXP-2026-005", "r2")  # read only

## 1. Code from the fork, packages, Drive API, run folders

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # before any CUDA work: repeatable cuBLAS
from google.colab import drive

drive.mount("/content/drive")  # RHUH exports are read through the mount; every write goes through the Drive API
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

from rhuh_synthesis import colab_packages  # the one package list; it imports nothing (a module, so no name clash)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *colab_packages.DATA, *colab_packages.MODEL], check=True)
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(commit)

from google.colab import auth
from googleapiclient.discovery import build

from rhuh_synthesis.drive_api import DriveRun

auth.authenticate_user()  # Colab asks once for permission to use the Drive API with this account
api = build("drive", "v3", cache_discovery=False)

assert MODEL in ("E0", "E0s", "E3m"), f"MODEL must be E0, E0s or E3m, got {MODEL!r}"
assert MODEL != "E0" or TRAIN_TAG is None, "MODEL = 'E0' is r2 and needs TRAIN_TAG = None"
assert MODEL == "E0" or TRAIN_TAG, f"MODEL = {MODEL!r} needs TRAIN_TAG = the RUN_TAG of its training run, e.g. '{MODEL}_r1'"
assert not SMOKE_FIRST or MODEL == "E0", "SMOKE_FIRST = True belongs to the first session (MODEL = 'E0'); set it to False"
LOCAL_RUN = Path(f"/content/e_eval_{RUN_TAG}")
assert not LOCAL_RUN.exists(), f"{LOCAL_RUN} exists: choose a new RUN_TAG or a fresh runtime"
settings = json.loads(Path("configs/experiments/EXP-2026-007.json").read_text(encoding="utf-8"))
if MODEL == "E0":
    weights_run = DriveRun.open(api, DATA_ON_DRIVE, ("stage5_runs", settings["runs"]["E0"]["weights"]["stage5_run_tag"]),
                                "/content/e_eval_weights", expect_in_base=("split", "export_v1"))
else:
    weights_run = DriveRun.open(api, OUT_BASE, ("EXP-2026-007", "train", TRAIN_TAG), "/content/e_eval_weights",
                                expect_in_base=("EXP-2026-001",))
# Base by path (shortcuts and folders shared with this account are followed), run folder below it
lumiere = DriveRun.open(api, LUMIERE_RUN[:-1], LUMIERE_RUN[-1:], "/content/e_eval_lumiere_r2", create=False)
run_drive = DriveRun.open(api, OUT_BASE, OUT_RUN, LOCAL_RUN, expect_in_base=("EXP-2026-001",), create=True)
access = run_drive.preflight()  # a real write, update, read and removal on Drive before any GPU time
print("free GiB on Drive:", None if access["free_gib"] is None else round(access["free_gib"], 1))
STARTED = time.perf_counter()


def say(message):
    print(f"[{time.perf_counter() - STARTED:6.0f} s] {message}", flush=True)

## 2. The model to evaluate, checked against its record, and the frozen encoders

In [ ]:
import numpy as np
import torch
from diffusers import AutoencoderKL, DDPMScheduler
from huggingface_hub import hf_hub_download
from transformers import CLIPTextModel, CLIPTokenizer

from rhuh_synthesis.e_series import state_dict_sha256
from rhuh_synthesis.model import reference_ddim_scheduler, scheduler_mismatches
from rhuh_synthesis.overfit import load_trained_unet
from rhuh_synthesis.provenance import code_version, config_sha256, environment
from rhuh_synthesis.stage3 import clip_encoder, load_reference_unet
from StableDiffusion.Our_UNet import UNet2DConditionModel

torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False
config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))  # frozen, read only
d2 = json.loads(Path("configs/experiments/EXP-2026-008_d2.json").read_text(encoding="utf-8"))  # frozen, read only
source = json.loads(weights_run.fetch("record.json").read_text(encoding="utf-8"))  # by ID, checked by MD5
WEIGHTS = weights_run.fetch(source["weights"]["file"])
unet4, load_report = load_reference_unet(UNet2DConditionModel, UNet2DConditionModel.load_config(SD_REPO, subfolder="unet"),
                                         hf_hub_download(MSF_REPO, MSF_FILE))
assert unet4 is not None, "Checkpoint keys do not match the vendored UNet: stop and report"
unet, weights_report = load_trained_unet(unet4, WEIGHTS, device="cuda")
del unet4
torch.cuda.empty_cache()
assert weights_report["sha256"] == source["weights"]["sha256"], "The weights differ from their training record: stop and report"
if MODEL != "E0":
    assert source["run"] == MODEL and source["weights"]["step"] == settings["training"]["evaluated_checkpoint_step"], \
        "The training record belongs to another run or step: stop and report"
vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae").to("cuda").eval().requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder").to("cuda").eval().requires_grad_(False)
ddim = reference_ddim_scheduler()
assert scheduler_mismatches(ddim) == []
noise_scheduler = DDPMScheduler.from_pretrained(SD_REPO, subfolder="scheduler")
encode_text = clip_encoder(tokenizer, text_encoder, "cuda")
MODELS = {"unet": unet, "vae": vae, "scheduler": ddim, "noise_scheduler": noise_scheduler, "encode_text": encode_text,
          "codec": config["representation"]["mask_codec"], "crop_mm": config["representation"]["crop_mm"],
          "out_px": config["representation"]["out_px"], "num_inference_steps": config["sampling"]["num_inference_steps"],
          "guidance_scale": config["sampling"]["guidance_scale"], "device": "cuda"}
record = {
    "experiment": "EXP-2026-007", "part": "E-eval", "model": MODEL, "train_tag": TRAIN_TAG, "run_tag": RUN_TAG,
    "commit": commit, "code_version": code_version(WORK), "settings_sha256": config_sha256(settings),
    "exp001_config_sha256": config_sha256(config), "d2_config_sha256": config_sha256(d2), "environment": environment(),
    "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "weights": weights_report, "seeds": settings["evaluation"]["seeds"],
    "training_lineage_sha256": source.get("lineage_sha256"),
}
say(f"models ready; weights {weights_report['sha256'][:12]}")

## 3. Held-out patients (V1 to V7) and the training exports for the bank

In [ ]:
from rhuh_synthesis.e_series import accepted_development_pairs, heldout_labels, rhuh_training_members
from rhuh_synthesis.lumiere_scaffold_check import accepted_training_pairs, verified_split
from rhuh_synthesis.pilot import accepted_patients, local_copies

PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
members = rhuh_training_members(split, policy)
val_dir = PRIVATE / "export_v1/val"
val_patients = accepted_patients(split["roles"]["val"]["patients"], val_dir)
assert not set(val_patients) & set(members) and not set(val_patients) & set(split["roles"]["test"]["patients"])
for name in ("run.json", "split.json", "registration_qc.csv", "summary.json"):
    lumiere.fetch(f"outputs/{name}")
outputs = Path("/content/e_eval_lumiere_r2/outputs")
lumiere_split, lumiere_split_sha, _ = verified_split(outputs)
lumiere.fetch_folder("outputs/export/development")  # scored only
dev_pairs, dev_counts = accepted_development_pairs(outputs, lumiere_split, lumiere_split_sha)
expected = settings["evaluation"]["expected_heldout"]
assert len(val_patients) == expected["RHUH"] and dev_counts["accepted"] == expected["LUMIERE"], "Held-out counts differ from the card"
train_pairs = []
if SMOKE_FIRST or MODEL == "E3m":
    lumiere.fetch_folder("outputs/export/train")  # the smoke's LUMIERE item and E3m's own memorization bank
    train_pairs, train_counts = accepted_training_pairs(outputs, lumiere_split, lumiere_split_sha)
    assert train_counts["accepted"] == settings["lumiere_source"]["expected_train_accepted"]
LABELS = heldout_labels(val_patients, [path.stem for path, _ in dev_pairs])
rhuh_paths = local_copies({p: val_dir / f"{p}.npz" for p in val_patients}, "/content/e_eval_val")
dev_paths = {path.stem: path for path, _ in dev_pairs}
train_paths = local_copies({p: next(PRIVATE.glob(f"export_v1/train*/{p}.npz")) for p in members}, "/content/e_eval_train")


def load_arrays(path):
    with np.load(path) as data:
        return {key: data[key] for key in data.files}


def heldout_arrays(label):
    cohort, item = LABELS[label]
    return cohort, load_arrays(rhuh_paths[item] if cohort == "RHUH" else dev_paths[item])


say(f"held out: {expected}; RHUH training patients for the bank: {len(members)}")

## 4. First session only: the mixed-batch smoke (decision 9)

In [ ]:
if SMOKE_FIRST:
    from rhuh_synthesis.representation import crop_forward
    from rhuh_synthesis.training import build_feature_cache, build_image_only_cache, merge_caches, train_step
    from rhuh_synthesis.working_grid import to_image_only_grid, to_working_grid

    before = state_dict_sha256(unet)
    grid_size = dict(crop_mm=MODELS["crop_mm"], out_px=MODELS["out_px"])
    _, grid_r = to_working_grid(load_arrays(train_paths[members[0]]), **grid_size)
    _, grid_l = to_image_only_grid(load_arrays(train_pairs[0][0]), **grid_size)
    middle = lambda grid: slice(len(grid["post_image"]) // 2 - 2, len(grid["post_image"]) // 2 + 2)
    cut = lambda grid: {k: v[middle(grid)] for k, v in grid.items() if np.ndim(v) == 3}
    smoke_cache = merge_caches(
        build_feature_cache({"R": cut(grid_r)}, vae, encode_text, config["text"], batch=4, device=MODELS["device"],
                            mask_codec=config["representation"]["mask_codec"]),
        build_image_only_cache({"L": cut(grid_l)}, vae, encode_text, cohort="LUMIERE", batch=4, device=MODELS["device"]),
        cohort="RHUH")
    seen = []
    hook = unet.register_forward_pre_hook(lambda module, args: seen.append(args[1].detach().cpu().clone()))
    unet.train()
    smoke = {}
    batches = {"labelled_only": [("R", 0, "negative"), ("R", 1, "negative")],
               "image_only": [("L", 0, "negative"), ("L", 1, "negative")],
               "mixed": [("R", 2, "negative"), ("L", 2, "negative")]}
    for name, items in batches.items():
        seen.clear()
        with torch.autocast(device_type=MODELS["device"], dtype=torch.bfloat16):
            loss = train_step(unet, smoke_cache, items, noise_scheduler, rng=np.random.default_rng(0),
                              generator=torch.Generator().manual_seed(0))
        loss.backward()
        grads = [p.grad for p in unet.parameters() if p.grad is not None]
        finite = bool(torch.isfinite(loss)) and all(bool(torch.isfinite(g).all()) for g in grads)
        absent = [k for k, (item, _, _) in enumerate(items) if item == "L"]
        partner_999 = all(int(seen[-1][len(items) + k]) == 999 for k in absent)
        smoke[name] = {"loss": float(loss.detach()), "finite": finite, "grad_tensors": len(grads),
                       "partner_rows_at_999": partner_999 if absent else None}
        unet.zero_grad(set_to_none=True)
    hook.remove()
    unet.eval()
    after = state_dict_sha256(unet)
    smoke["weights_unchanged"] = before == after
    record["smoke"] = smoke
    print("SMOKE", json.dumps(smoke, indent=1))
    assert all(v["finite"] and v["grad_tensors"] > 0 for k, v in smoke.items() if k != "weights_unchanged")
    assert smoke["image_only"]["partner_rows_at_999"] and smoke["mixed"]["partner_rows_at_999"]
    assert smoke["weights_unchanged"], "The smoke changed the weights: stop and report"
    del smoke_cache, grid_r, grid_l
    torch.cuda.empty_cache()

## 5. W4 on every held-out patient, scored from the saved stacks; W7 and held-out loss

In [ ]:
from rhuh_synthesis.e_eval import evaluate_patient, recompute_scores

ranks = {label: k for k, label in enumerate(l for l in LABELS if LABELS[l][0] == "RHUH")}
patients, kept = {}, {}
torch.cuda.reset_peak_memory_stats()
for label in LABELS:
    cohort, arrays = heldout_arrays(label)
    entry, inputs, paths = evaluate_patient(arrays, label=label, cohort=cohort, rank=ranks.get(label, 0), settings=settings,
                                            evaluation=config["evaluation"], text_config=config["text"], d2_cfg=d2,
                                            models=MODELS, out_dir=LOCAL_RUN, progress=say)
    differing = recompute_scores(arrays, inputs, paths, entry, config["evaluation"], crop_mm=MODELS["crop_mm"],
                                 out_px=MODELS["out_px"], d2_cfg=d2)
    assert differing == [], f"{label}: scores do not recompute from the saved stacks for seeds {differing}"
    patients[label], kept[label] = entry, (inputs, paths)
    say(f"{label} ({cohort}): primary MAE " + ", ".join(f"{v['primary_mae']:.4f}" for v in entry["w4"].values()))
    run_drive.sync()
record.update(patients=patients, items={label: list(item) for label, item in LABELS.items()},
              recomputed_identically=True, peak_gib=torch.cuda.max_memory_allocated() / 2**30)

## 6. Memorization alert against the training banks

In [ ]:
from rhuh_synthesis.e_eval import bank_from_grids, patient_alerts
from rhuh_synthesis.representation import crop_forward
from rhuh_synthesis.working_grid import to_image_only_grid, to_working_grid


def bank_grid(arrays, labelled=True):
    crop, grid = (to_working_grid if labelled else to_image_only_grid)(arrays, crop_mm=MODELS["crop_mm"], out_px=MODELS["out_px"])
    return {"post_image": grid["post_image"], "post_support": crop_forward(arrays["post_support"].astype(np.uint8), crop).astype(bool)}


rhuh_loaders = {p: (lambda p=p: bank_grid(load_arrays(train_paths[p]))) for p in members}  # one grid in memory at a time
common = bank_from_grids(rhuh_loaders)
banks = {"common_rhuh": tuple(t.to(MODELS["device"]) if torch.is_tensor(t) else t for t in common)}
if MODEL == "E3m":
    lumiere_loaders = {path.stem: (lambda path=path: bank_grid(load_arrays(path), labelled=False)) for path, _ in train_pairs}
    own = bank_from_grids({**rhuh_loaders, **lumiere_loaders})
    banks["own_rhuh_lumiere"] = tuple(t.to(MODELS["device"]) if torch.is_tensor(t) else t for t in own)
rates, matches = {}, []
for label in LABELS:
    _, arrays = heldout_arrays(label)
    inputs, paths = kept[label]
    rates[label], found = patient_alerts(arrays, inputs, paths, banks, settings["evaluation"]["memorization"],
                                         crop_mm=MODELS["crop_mm"], out_px=MODELS["out_px"], device=MODELS["device"])
    matches.extend(dict(m, label=label) for m in found)
totals = {}
for by_bank in rates.values():
    for bank, entry in by_bank.items():
        total = totals.setdefault(bank, {"bank_slices": entry["bank_slices"]})
        for kind in ("generated", "copy", "scaffold"):
            t = total.setdefault(kind, {"queries": 0, "scored": 0, "alerts": 0})
            for key in t:
                t[key] += entry[kind][key]
record["memorization"] = {"totals": totals, "by_patient": rates}
record["memorization_matches_private"] = matches
del banks, common
torch.cuda.empty_cache()
say(f"memorization: {totals}")

## 7. The write-once record, the summary and, for E3m, the decision

In [ ]:
from rhuh_synthesis.e_eval import model_summary, series_decision
from rhuh_synthesis.manifest import write_once_json

record["seconds"] = time.perf_counter() - STARTED
path, digest = write_once_json(LOCAL_RUN / "e_eval_record.json", record)
report = run_drive.sync()
say(f"record {digest[:12]} written; Drive upload: {report}")
summary = model_summary(record)
summary["memorization"] = record["memorization"]["totals"]
if record.get("smoke"):
    summary["smoke"] = record["smoke"]
decision = None
others = {}
for name, tag in EVAL_TAGS.items():
    if tag:
        other = DriveRun.open(api, OUT_BASE, ("EXP-2026-007", "eval", tag), f"/content/e_eval_other_{tag}",
                              expect_in_base=("EXP-2026-001",))
        others[name] = json.loads(other.fetch("e_eval_record.json").read_text(encoding="utf-8"))
        assert others[name]["model"] == name, f"{tag} is not an evaluation of {name}"
if MODEL == "E0s" and "E0" in others:
    gap = {label: abs(np.mean([v["primary_mae"] for v in record["patients"][label]["w4"].values()])
                      - np.mean([v["primary_mae"] for v in others["E0"]["patients"][label]["w4"].values()]))
           for label in record["patients"]}
    summary["seed_comparator_per_patient"] = {label: round(float(v), 4) for label, v in gap.items()}
if MODEL == "E3m" and {"E0", "E0s"} <= set(others):
    decision = series_decision(others["E0"], others["E0s"], record, settings["evaluation"])

## 8. Confirm the upload and print

In [ ]:
assert run_drive.confirm(), f"Not every file is confirmed on Drive. The originals are in {LOCAL_RUN}: keep this session open and report"
print("Every file confirmed by the Drive server: MyDrive/" + "/".join(OUT_BASE + OUT_RUN))
print("SUMMARY")
print(json.dumps(summary, indent=1))
if decision is not None:
    print("DECISION")
    print(json.dumps(decision, indent=1))

## What to send back

Copy everything printed after `SUMMARY` (and `DECISION` in E3m's session), and the `SMOKE` block in the first session.
They hold scores and counts by V1 to V7, no patient name.

If a cell stops, send the error message. The run stops on purpose when a count, a split, an export, the weights' hash,
the smoke or the recomputation from the saved stacks does not match.